# PalmGuard AI — MobileNetV2 Training (Google Colab)

Runs the pipeline in `ai/src/` end-to-end: Data Cleaning -> Split -> Preprocessing -> Augmentation -> MobileNetV2 Transfer Learning -> Fine-tuning -> Evaluation -> Export. See `docs/AI_MODEL.md` for the full spec.

**Before running:** upload/mount a `PalmGuard/ai` folder containing `config.py`, `src/`, `requirements.txt`, and a populated `data/raw/<CLASS_NAME>/` per class (`Healthy`, `Brown_Spot`, `Insect_Damage`, `Non_Palm`) — this notebook does not include any images itself.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Point this at wherever the ai/ folder (config.py, src/, data/raw/...) lives in your Drive.
AI_DIR = '/content/drive/MyDrive/PalmGuard/ai'

%cd {AI_DIR}

In [ ]:
# Colab already ships a GPU-enabled TensorFlow build. ai/requirements.txt pins
# tensorflow-cpu (Windows has no native GPU support for tensorflow>=2.11 — see
# the comment at the top of that file), so skip that one line here and install
# the rest.
!grep -v '^tensorflow-cpu' requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 1. Data Cleaning
Checklist from `docs/AI_MODEL.md#data-cleaning` — corrupt files, duplicates, blur, class balance. Fix anything flagged here before training; a bad dataset produces a bad model regardless of architecture.

In [ ]:
from src.data_cleaning import validate_dataset, check_class_balance

reports = validate_dataset()
print(f"{'Class':<15}{'Images':>8}{'Corrupt':>9}{'Blurry':>8}{'Dup groups':>12}")
for name, r in reports.items():
    print(f"{name:<15}{r.image_count:>8}{len(r.corrupt_files):>9}{len(r.blurry_files):>8}{len(r.duplicate_groups):>12}")

imbalance = check_class_balance(reports)
if imbalance:
    print('\n\u26a0 Class imbalance:')
    for line in imbalance:
        print(' -', line)

## 2. Train
Splits the dataset (group-aware by source tree — see `docs/AI_MODEL.md#train--validation--test`), then runs Phase 1 (frozen MobileNetV2 base + head) and Phase 2 (fine-tuning, top layers unfrozen) back to back, per `config.py`.

In [ ]:
from src.train import train

model_path = train()
print('Exported model to:', model_path)

## 3. Evaluation report
`train()` already writes `models/evaluation-{MODEL_VERSION}.json` (Accuracy/Precision/Recall/F1/ROC AUC/confusion matrix per class) and `models/history-{MODEL_VERSION}.json` (per-epoch loss/accuracy for both training phases). Load and inspect them here.

In [ ]:
import json
import config

with open(config.MODELS_DIR / f'evaluation-{config.MODEL_VERSION}.json', encoding='utf-8') as f:
    evaluation = json.load(f)

print(f"Test accuracy: {evaluation['accuracy']*100:.2f}%\n")
for name, m in evaluation['per_class'].items():
    print(f"{name:<15} correct={m['correct']:>3}/{m['total']:<3} "
          f"precision={m['precision']:.3f} recall={m['recall']:.3f} "
          f"f1={m['f1']:.3f} roc_auc={m.get('roc_auc', float('nan')):.3f}")

In [ ]:
import matplotlib.pyplot as plt

with open(config.MODELS_DIR / f'history-{config.MODEL_VERSION}.json', encoding='utf-8') as f:
    history = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for phase in ('head', 'fine_tune'):
    axes[0].plot(history[phase]['accuracy'], label=f'{phase} train')
    axes[0].plot(history[phase]['val_accuracy'], label=f'{phase} val')
    axes[1].plot(history[phase]['loss'], label=f'{phase} train')
    axes[1].plot(history[phase]['val_loss'], label=f'{phase} val')
axes[0].set_title('Accuracy'); axes[0].legend()
axes[1].set_title('Loss'); axes[1].legend()
plt.show()

In [ ]:
import numpy as np

cm = np.array(evaluation['confusion_matrix'])
class_names = evaluation['class_names']

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, cm[i, j], ha='center', va='center')
plt.colorbar(im)
plt.show()